# Example 7.7.1 / 1: SOS2: One Point or Two Adjacent Points

Companion notebook for **7.7.1 SOS1 and SOS2 Special Ordered Sets** in the AMPL lab Documentation, section "SOS2: One Point or Two Adjacent Points". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## SOS2: One Point or Two Adjacent Points

An SOS2 group permits at most two nonzero members, and two nonzeros must be adjacent in `.ref`
order. The standard interpolation formulation uses nonnegative weights $\lambda_k$:

$$
\sum_k \lambda_k = 1, \qquad
x = \sum_k \bar{x}_k\lambda_k, \qquad
y = \sum_k \bar{y}_k\lambda_k.
$$

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
suffix sosno integer IN;          # SOS group: positive for SOS1, negative for SOS2
suffix ref IN;                    # order of the members within a group

set POINTS ordered;               # breakpoints of the curve
param x_value {POINTS};           # x coordinate of the breakpoint
param y_value {POINTS};           # y coordinate of the breakpoint

# interpolation weights; SOS2 allows two adjacent positive weights
var Lambda {k in POINTS} >= 0, <= 1,
    suffix sosno -1,
    suffix ref ord(k);
var x;                            # interpolated x
var y;                            # interpolated y

# weights form a convex combination
subject to Convex_Weights:
    sum {k in POINTS} Lambda[k] = 1;
# x from the breakpoint x coordinates
subject to Interpolate_X:
    x = sum {k in POINTS} x_value[k] * Lambda[k];
# y from the matching y coordinates
subject to Interpolate_Y:
    y = sum {k in POINTS} y_value[k] * Lambda[k];

param x_target;                   # where to read the curve
subject to AtTarget: x = x_target;
maximize Height: y;               # highest y the weights allow at x_target

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["POINTS"] = ["p1", "p2", "p3", "p4"]
ampl.param["x_value"] = {"p1": 0, "p2": 2, "p3": 4, "p4": 6}
ampl.param["y_value"] = {"p1": 0, "p2": 6, "p3": 2, "p4": 6}
ampl.param["x_target"] = 4

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="gurobi")
print(ampl.solve_result, "y =", round(ampl.var["y"].value(), 4))
print({k: round(v, 4) for k, v in ampl.var["Lambda"].to_dict().items()})

solved y = 2.0
{'p1': 0, 'p2': 0.0, 'p3': 1.0, 'p4': 0}


### Pure AMPL: data.dat and model.run

The curve passes through `(4, 2)`, and Gurobi returns `y = 2` with all weight on `p3`. Without
the SOS2 restriction, weights 0.5 on `p2` and `p4` would also give `x = 4` but `y = 6`, a point
on the convex hull of the breakpoints rather than on the curve.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: POINTS: x_value y_value :=
  p1  0  0
  p2  2  6
  p3  4  2
  p4  6  6 ;
param x_target := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gurobi;
option solver_msg 0;
solve;
printf "%s y = %.4f\n", solve_result, y;

solved y = 2.0000


### Another way in amplpy

Not every solver accepts the declaration. HiGHS has no SOS constraints, and its MP interface does
not convert a general SOS2 into binaries, so the solve fails with the same `model.mod` and
`data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["POINTS"] = ["p1", "p2", "p3", "p4"]
ampl.param["x_value"] = {"p1": 0, "p2": 2, "p3": 4, "p4": 6}
ampl.param["y_value"] = {"p1": 0, "p2": 6, "p3": 2, "p4": 6}
ampl.param["x_target"] = 4

In [11]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)

HiGHS 1.14.0: MIPFlatConverter: MIP conversion of general SOS2 (not used to encode a pl): not implemented
failure
